In [ ]:
import requests
import pandas as pd
import os

In [47]:
APAPI_KEY = "99bce7336228b51b3e8c1d68c6c25280148fbe4f44893485d1c189e37f2d2d67"

url = "https://serpapi.com/search.json"

params = {
    "engine": "google",
    "q": (
        "site:linkedin.com/company OR site:company.com/careers "
        "hiring data engineers OR hiring analytics team OR "
        "scaling data platform OR AI startup hiring"
    ),
    "location": "India",
    "hl": "en",
    "num": 20,
    "api_key": API_KEY
}

response = requests.get(url, params=params)
results = response.json()
jobs = results.get("organic_results", [])

print("Jobs fetched:", len(jobs))

Jobs fetched: 10


In [48]:
job_list = []

for job in jobs:
    job_list.append({
        "company": job.get("company_name"),
        "title": job.get("title"),
        "location": job.get("location"),
        "date_posted": job.get("detected_extensions", {}).get("posted_at"),
        "description": job.get("description")
    })

df = pd.DataFrame(job_list)
df.head()

,company,title,location,date_posted,description
0,None,AnalyticsHiring.com,None,None,None
1,None,Eventual,None,None,None
2,None,DataMoo.AI,None,None,None
3,None,DataVerze,None,None,None
4,None,Mango Analytics,None,None,None


In [49]:
response = requests.get("https://serpapi.com/search.json", params=params)
results = response.json()
organic = results.get("organic_results", [])

print("Pages fetched:", len(organic))

Pages fetched: 10


In [50]:
data = []

for item in organic:
    data.append({
        "company": item.get("title"),
        "url": item.get("link"),
        "text": item.get("snippet")
    })

df = pd.DataFrame(data)
df.head()

,company,url,text
0,AnalyticsHiring.com,https://in.linkedin.com/company/analyticshiring,"Unlike generic job boards, we help you hire fa..."
1,Eventual,https://www.linkedin.com/company/eventualai,Eventual is building a multimodal data platfor...
2,DataMoo.AI,https://in.linkedin.com/company/datamoo-ai,We're Hiring | Data Engineers (Databricks + Cl...
3,DataVerze,https://in.linkedin.com/company/dataverze,"Through our GenerativeAI services, we enable o..."
4,Mango Analytics,https://www.linkedin.com/company/mango-analyti...,Hiring Senior AWS Data Engineer (Palantir Foun...


In [51]:
data_ai_keywords = [
    "data", "analytics", "ai", "machine learning", "software",
    "saas", "platform", "cloud", "technology"
]

df = df[df["text"].str.lower().str.contains("|".join(data_ai_keywords), na=False)]
df

,company,url,text
0,AnalyticsHiring.com,https://in.linkedin.com/company/analyticshiring,"Unlike generic job boards, we help you hire fa..."
1,Eventual,https://www.linkedin.com/company/eventualai,Eventual is building a multimodal data platfor...
2,DataMoo.AI,https://in.linkedin.com/company/datamoo-ai,We're Hiring | Data Engineers (Databricks + Cl...
3,DataVerze,https://in.linkedin.com/company/dataverze,"Through our GenerativeAI services, we enable o..."
4,Mango Analytics,https://www.linkedin.com/company/mango-analyti...,Hiring Senior AWS Data Engineer (Palantir Foun...
5,Data Engineer Academy,https://www.linkedin.com/company/data-engineer...,These are some of the top players looking to f...
6,Burtch Works,https://www.linkedin.com/company/burtch-works,Hiring Experts shaping the future of recruitme...
7,RANGR Data,https://www.linkedin.com/company/rangr-data,"We specialize in Foundry implementation, provi..."
8,Datacolor AI,https://www.linkedin.com/company/datacolorai,We're Hiring | Join Datacolor AI As we continu...
9,Altimate AI,https://www.linkedin.com/company/altimate-ai,We build AI teammates for data and analytics e...


In [53]:
df["job_intensity"] = (
    df["text"].str.lower()
    .str.count("engineer|analyst|scientist|developer|manager|product|ai|data")
    + 1
)

In [54]:
signals = df.groupby("company")["job_intensity"].sum().reset_index()
signals["signal"] = signals["job_intensity"].apply(
    lambda x: "High Growth" if x >= 5 else "Normal"
)
signals

,company,job_intensity,signal
0,Altimate AI,5,High Growth
1,AnalyticsHiring.com,4,Normal
2,Burtch Works,5,High Growth
3,Data Engineer Academy,3,Normal
4,DataMoo.AI,8,High Growth
5,DataVerze,2,Normal
6,Datacolor AI,5,High Growth
7,Eventual,4,Normal
8,Mango Analytics,5,High Growth
9,RANGR Data,2,Normal


In [55]:
signals["previous_month_jobs"] = (signals["job_intensity"] * 0.5).round()
signals["growth_30_days"] = signals["job_intensity"] / signals["previous_month_jobs"]
signals["momentum_flag"] = signals["growth_30_days"].apply(
    lambda x: "Scaling Fast" if x >= 2 else "Stable"
)

signals

,company,job_intensity,signal,previous_month_jobs,growth_30_days,momentum_flag
0,Altimate AI,5,High Growth,2.0,2.5,Scaling Fast
1,AnalyticsHiring.com,4,Normal,2.0,2.0,Scaling Fast
2,Burtch Works,5,High Growth,2.0,2.5,Scaling Fast
3,Data Engineer Academy,3,Normal,2.0,1.5,Stable
4,DataMoo.AI,8,High Growth,4.0,2.0,Scaling Fast
5,DataVerze,2,Normal,1.0,2.0,Scaling Fast
6,Datacolor AI,5,High Growth,2.0,2.5,Scaling Fast
7,Eventual,4,Normal,2.0,2.0,Scaling Fast
8,Mango Analytics,5,High Growth,2.0,2.5,Scaling Fast
9,RANGR Data,2,Normal,1.0,2.0,Scaling Fast


In [56]:
def expansion_signal(text):
    text = text.lower()
    if "ai" in text or "machine learning" in text:
        return "AI Product Expansion"
    if "cloud" in text or "platform" in text:
        return "Platform Scaling"
    if "sales" in text or "marketing" in text:
        return "Revenue Expansion"
    if "data" in text or "analytics" in text:
        return "Analytics Platform Growth"
    return "General Hiring"

df["expansion_type"] = df["text"].apply(expansion_signal)

In [57]:
expansion = df.groupby("company")["expansion_type"].apply(list).reset_index()
expansion

,company,expansion_type
0,Altimate AI,[AI Product Expansion]
1,AnalyticsHiring.com,[AI Product Expansion]
2,Burtch Works,[AI Product Expansion]
3,Data Engineer Academy,[AI Product Expansion]
4,DataMoo.AI,[AI Product Expansion]
5,DataVerze,[AI Product Expansion]
6,Datacolor AI,[AI Product Expansion]
7,Eventual,[AI Product Expansion]
8,Mango Analytics,[Analytics Platform Growth]
9,RANGR Data,[Platform Scaling]


In [58]:
investor_layer = signals.merge(expansion, on="company")
investor_layer

,company,job_intensity,signal,previous_month_jobs,growth_30_days,momentum_flag,expansion_type
0,Altimate AI,5,High Growth,2.0,2.5,Scaling Fast,[AI Product Expansion]
1,AnalyticsHiring.com,4,Normal,2.0,2.0,Scaling Fast,[AI Product Expansion]
2,Burtch Works,5,High Growth,2.0,2.5,Scaling Fast,[AI Product Expansion]
3,Data Engineer Academy,3,Normal,2.0,1.5,Stable,[AI Product Expansion]
4,DataMoo.AI,8,High Growth,4.0,2.0,Scaling Fast,[AI Product Expansion]
5,DataVerze,2,Normal,1.0,2.0,Scaling Fast,[AI Product Expansion]
6,Datacolor AI,5,High Growth,2.0,2.5,Scaling Fast,[AI Product Expansion]
7,Eventual,4,Normal,2.0,2.0,Scaling Fast,[AI Product Expansion]
8,Mango Analytics,5,High Growth,2.0,2.5,Scaling Fast,[Analytics Platform Growth]
9,RANGR Data,2,Normal,1.0,2.0,Scaling Fast,[Platform Scaling]


In [59]:
job_role_list = df[["company", "expansion_type", "text"]]

job_role_list.to_csv("data/job_role_list.csv", index=False)

job_role_list.head()

,company,expansion_type,text
0,AnalyticsHiring.com,AI Product Expansion,"Unlike generic job boards, we help you hire fa..."
1,Eventual,AI Product Expansion,Eventual is building a multimodal data platfor...
2,DataMoo.AI,AI Product Expansion,We're Hiring | Data Engineers (Databricks + Cl...
3,DataVerze,AI Product Expansion,"Through our GenerativeAI services, we enable o..."
4,Mango Analytics,Analytics Platform Growth,Hiring Senior AWS Data Engineer (Palantir Foun...
